In [100]:
# Import necessary libraries
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import numpy as np

print("Libraries imported successfully.")


Libraries imported successfully.


In [150]:
import requests
from shapely.geometry import Point

# Overpass query for Spätis & corner stores in Berlin
query = """
[out:json];
(
  node[shop=convenience](52.4,13.2,52.7,13.6);
  node[shop=kiosk](52.4,13.2,52.7,13.6);
);
out body;
"""

# Send request
url = "https://overpass-api.de/api/interpreter"
resp = requests.post(url, data=query, timeout=180)
resp.raise_for_status()
data = resp.json()

# Normalize JSON to pandas dataframe
elements = data.get("elements", [])
df = pd.json_normalize(elements)

# Ensure lat/lon exist
df = df[df['lat'].notna() & df['lon'].notna()]

# Convert to GeoDataFrame
spatis_gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df['lon'].astype(float), df['lat'].astype(float))],
    crs="EPSG:4326"
)

# Minimal cleanup and renaming to match schema
spatis = spatis_gdf.rename(columns={
    'tags.name': 'name',
    'tags.brand': 'brand',
    'tags.operator': 'operator',
    'tags.opening_hours': 'opening_hours',
    'tags.phone': 'phone',
    'tags.website': 'website',
    'tags.source': 'source',
    'lat': 'latitude',
    'lon': 'longitude',
    'type': 'osm_type',
    'id': 'id'  # Keep OSM ID as 'id'
}).copy()

print("SPATIS fetched and loaded. Records:", len(spatis))
spatis.head(5)


SPATIS fetched and loaded. Records: 1607


,osm_type,id,latitude,longitude,tags.addr:city,tags.addr:country,tags.addr:housenumber,tags.addr:postcode,tags.addr:street,tags.addr:suburb,...,tags.post_office:opening_hours,tags.money_transfer,tags.mobile,tags.fuel:HGV_diesel,tags.fuel:octane_100,tags.post_office:id_check,tags.name:ko,tags.public_transport,tags.ticket,geometry
0,node,26867411,52.501974,13.294496,Berlin,DE,14,10711,Heilbronner Straße,Halensee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.2945 52.50197)
1,node,29997723,52.508370,13.280947,Berlin,DE,8-10,14057,Messedamm,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.28095 52.50837)
2,node,63253672,52.499322,13.296118,Berlin,DE,39,10711,Joachim-Friedrich-Straße,Halensee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.29612 52.49932)
3,node,253616592,52.511047,13.462698,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.4627 52.51105)
4,node,266629404,52.509209,13.587500,Berlin,DE,45,12621,Mädewalder Weg,Kaulsdorf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (13.5875 52.50921)


In [156]:
print("Columns in spatis:", spatis.columns.tolist())

Columns in spatis: ['osm_type', 'id', 'latitude', 'longitude', 'tags.addr:city', 'tags.addr:country', 'tags.addr:housenumber', 'tags.addr:postcode', 'tags.addr:street', 'tags.addr:suburb', 'tags.amenity', 'tags.check_date:opening_hours', 'tags.compressed_air', 'tags.fuel:adblue', 'tags.fuel:biodiesel', 'tags.fuel:diesel', 'tags.fuel:e10', 'tags.fuel:octane_95', 'tags.fuel:octane_98', 'name', 'opening_hours', 'operator', 'tags.shop', 'tags.wheelchair', 'brand', 'tags.brand:wikidata', 'tags.brand:wikipedia', 'tags.fuel:GTL_diesel', 'tags.fuel:biogas', 'tags.fuel:cng', 'tags.fuel:lpg', 'tags.fuel:octane_102', 'tags.surveillance', 'website', 'tags.check_date', 'tags.dog', 'tags.email', 'tags.fax', 'phone', 'tags.start_date', 'tags.indoor_seating', 'tags.organic', 'tags.outdoor_seating', 'tags.smoking', 'tags.opening_hours:signed', 'tags.diet:halal', 'tags.level', 'tags.payment:credit_cards', 'tags.payment:debit_cards', 'tags.payment:apple_pay', 'tags.payment:cards', 'tags.payment:cash', 't

In [164]:
# Explore all columns and get summary statistics
spatis.describe(include="all")


,osm_type,id,latitude,longitude,tags.addr:city,tags.addr:country,tags.addr:housenumber,tags.addr:postcode,tags.addr:street,tags.addr:suburb,...,tags.mobile,tags.fuel:HGV_diesel,tags.fuel:octane_100,tags.post_office:id_check,tags.name:ko,tags.public_transport,tags.ticket,geometry,neighborhood_id,neighborhood_name
count,1607,1.607000e+03,1607.000000,1607.000000,686,489,795,707,829,475,...,2,1,1,1,1,1,1,1607,1582,1582
unique,1,NaN,NaN,NaN,6,1,233,149,439,53,...,2,1,1,1,1,1,1,1605,73,73
top,node,NaN,NaN,NaN,Berlin,DE,1,10245,Karl-Marx-Straße,Prenzlauer Berg,...,+491784027368,yes,yes,yes,삼일상사,service_center,public_transport,POINT (13.3129938 52.5017546),0801,Neukölln
freq,1607,NaN,NaN,NaN,677,489,19,33,14,59,...,1,1,1,1,1,1,1,2,158,158
mean,NaN,5.370249e+09,52.512150,13.396384,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,3.787071e+09,0.040233,0.068754,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,2.686741e+07,52.401392,13.200417,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,1.983339e+09,52.488105,13.347769,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,4.520971e+09,52.510031,13.402107,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,8.332127e+09,52.540426,13.439757,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [165]:
# Check missing values in each column
missing_count = spatis.isna().sum().sort_values(ascending=False)

# List columns where missing values are greater than 200
print(missing_count[missing_count > 200])


tags.sells:tobacco                1606
tags.note:de                      1606
tags.reusable_packaging:accept    1606
tags.wikidata                     1606
tags.wikipedia                    1606
                                  ... 
tags.addr:postcode                 900
opening_hours                      889
tags.addr:housenumber              812
tags.addr:street                   778
tags.wheelchair                    604
Length: 244, dtype: int64


In [166]:
# Check unique values in the 'brand' column
spatis['brand'].value_counts()



brand
REWE To Go                                  16
ServiceStore DB                             15
DHL                                          4
Yorma's                                      2
Total                                        2
Spar                                         2
DPD                                          1
JET                                          1
Shell Shop                                   1
Shell                                        1
Weltladen                                    1
Aral                                         1
Deutsche Post                                1
EDEKA                                        1
Elan                                         1
Lycamobile                                   1
Deutsche Post;DHL;Postbank;Western Union     1
Agip                                         1
Hermes                                       1
Edeka                                        1
TotalEnergies                                1
Name: c

In [167]:
# expand all columns to see more details
pd.set_option('display.max_columns', None)

print(spatis_gdf.head(3))


   type        id        lat        lon tags.addr:city tags.addr:country  \
0  node  26867411  52.501974  13.294496         Berlin                DE   
1  node  29997723  52.508370  13.280947         Berlin                DE   
2  node  63253672  52.499322  13.296118         Berlin                DE   

  tags.addr:housenumber tags.addr:postcode          tags.addr:street  \
0                    14              10711        Heilbronner Straße   
1                  8-10              14057                 Messedamm   
2                    39              10711  Joachim-Friedrich-Straße   

  tags.addr:suburb tags.amenity tags.check_date:opening_hours  \
0         Halensee         fuel                    2022-04-01   
1              NaN         fuel                           NaN   
2         Halensee          NaN                           NaN   

  tags.compressed_air tags.fuel:adblue tags.fuel:biodiesel tags.fuel:diesel  \
0                 yes              yes                 yes        

In [198]:
# Modelling & Planning
# Export SPATIS GeoDataFrame to the Downloads/Mapping folder

# Export as GeoJSON
spatis_gdf.to_file(
    "/Users/harrisongoodman/Downloads/spatis_raw.geojson",
    driver="GeoJSON"
)

# Export as CSV without geometry
spatis_gdf.drop(columns="geometry").to_csv(
    "/Users/harrisongoodman/Downloads/spatis_raw.csv",
    index=False
)

print("Raw SPATIS exported. Records:", len(spatis_gdf))


Raw SPATIS exported. Records: 1607


In [237]:
# Load districts
districts = gpd.read_file("/Users/harrisongoodman/Downloads/bezirksgrenzen.geojson")

# Rename correctly
districts = districts.rename(columns={
    "Gemeinde_schluessel": "district_id",
    "Gemeinde_name": "district_name"
})

# Keep only the necessary columns
districts = districts[["district_id", "district_name", "geometry"]].copy()

# Convert types
districts["district_id"] = districts["district_id"].astype(str)

print(districts.head())





  district_id               district_name  \
0         012               Reinickendorf   
1         004  Charlottenburg-Wilmersdorf   
2         009            Treptow-Köpenick   
3         003                      Pankow   
4         008                    Neukölln   

                                            geometry  
0  MULTIPOLYGON (((13.32074 52.6266, 13.32045 52....  
1  MULTIPOLYGON (((13.32111 52.52446, 13.32103 52...  
2  MULTIPOLYGON (((13.57925 52.39083, 13.57958 52...  
3  MULTIPOLYGON (((13.50481 52.6196, 13.50467 52....  
4  MULTIPOLYGON (((13.45832 52.48569, 13.45823 52...  


In [238]:
# Spatial join: assign each SPATIS point to a district
spatis = gpd.sjoin(
    spatis,
    districts[["district_id", "district_name", "geometry"]],
    how="left",
    predicate="intersects"
).drop(columns=["index_right"], errors="ignore")



In [239]:
# Load neighborhoods
neighborhoods = gpd.read_file("/Users/harrisongoodman/Downloads/lor_ortsteile.geojson")

# Correct: rename first
neighborhoods = neighborhoods.rename(columns={
    "spatial_name": "neighborhood_id",      # ID
    "OTEIL": "neighborhood_name"            # name
})

# Now select the correct columns
neighborhoods = neighborhoods[["neighborhood_id", "neighborhood_name", "geometry"]].copy()

# Convert ID to string
neighborhoods["neighborhood_id"] = neighborhoods["neighborhood_id"].astype(str)

# CRS fix to match SPATIS
neighborhoods = neighborhoods.to_crs(spatis.crs)

print(neighborhoods.head())

  neighborhood_id neighborhood_name  \
0            0101             Mitte   
1            0102            Moabit   
2            0103      Hansaviertel   
3            0104        Tiergarten   
4            0105           Wedding   

                                            geometry  
0  POLYGON ((13.41649 52.52696, 13.41635 52.52702...  
1  POLYGON ((13.33884 52.51974, 13.33884 52.51974...  
2  POLYGON ((13.34322 52.51557, 13.34323 52.51557...  
3  POLYGON ((13.36879 52.49878, 13.36891 52.49877...  
4  POLYGON ((13.34656 52.53879, 13.34664 52.53878...  


In [240]:
# Spatial join: assign each SPATIS point to a district
spatis = gpd.sjoin(
    spatis,
    districts[["district_id", "district_name", "geometry"]],
    how="left",
    predicate="intersects"
).drop(columns=["index_right"], errors="ignore")

print("Spatial join with districts complete. Records:", len(spatis))
spatis.head(5)


Spatial join with districts complete. Records: 1606


,id,name,brand,operator,latitude,longitude,neighborhood_id,neighborhood_name,openinghours,phone,website,geometry,address,district_id_left,district_name_left,district_id_right,district_name_right
0,26867411,Bavaria petrol,NaN,Bavaria Petrol,52.501974,13.294496,0407,Halensee,Mo-Fr 07:00-22:00; Sa 08:00-22:00,NaN,NaN,POINT (13.2945 52.50197),,004,Charlottenburg-Wilmersdorf,004,Charlottenburg-Wilmersdorf
1,29997723,Aral,Aral,Anne Notzke,52.508370,13.280947,0405,Westend,24/7,NaN,https://tankstelle.aral.de/tankstelle/berlin/m...,POINT (13.28095 52.50837),,004,Charlottenburg-Wilmersdorf,004,Charlottenburg-Wilmersdorf
2,63253672,Späti Joe,NaN,NaN,52.499322,13.296118,0407,Halensee,24/7,NaN,NaN,POINT (13.29612 52.49932),,004,Charlottenburg-Wilmersdorf,004,Charlottenburg-Wilmersdorf
3,253616592,Mein Markt Pham,NaN,NaN,52.511047,13.462698,0201,Friedrichshain,Mo-Fr 08:00-20:00; Sa 08:00-19:00,NaN,NaN,POINT (13.4627 52.51105),,002,Friedrichshain-Kreuzberg,002,Friedrichshain-Kreuzberg
4,266629404,...nah und gut,EDEKA,Heinz Vollack,52.509209,13.587500,1003,Kaulsdorf,Mo-Fr 07:00-19:00; Sa 07:00-13:00; PH off,+49 30 5677706,NaN,POINT (13.5875 52.50921),,010,Marzahn-Hellersdorf,010,Marzahn-Hellersdorf


In [219]:
# ================================
# Spatial join: neighborhoods
# ================================
spatis = gpd.sjoin(
    spatis,
    neighborhoods[["neighborhood_id", "neighborhood_name", "geometry"]],
    how="left",
    predicate="intersects"
)

# Drop the index_right column added by sjoin
spatis = spatis.drop(columns=["index_right"], errors="ignore")

# Keep only the right-hand join columns and rename them
for col in ["neighborhood_id", "neighborhood_name"]:
    if f"{col}_right" in spatis.columns:
        spatis[col] = spatis[f"{col}_right"]

# Drop any leftover left/right duplicate columns
spatis = spatis.drop(columns=[
    "neighborhood_id_left", "neighborhood_id_right",
    "neighborhood_name_left", "neighborhood_name_right"
], errors="ignore")



In [241]:
# ================================
# Spatial join: districts
# ================================
spatis = gpd.sjoin(
    spatis,
    districts[["district_id", "district_name", "geometry"]],
    how="left",
    predicate="intersects"
)

# Drop the index_right column added by sjoin
spatis = spatis.drop(columns=["index_right"], errors="ignore")

# Keep only the right-hand join columns and rename them
for col in ["district_id", "district_name"]:
    if f"{col}_right" in spatis.columns:
        spatis[col] = spatis[f"{col}_right"]

# Drop any leftover left/right duplicate columns
spatis = spatis.drop(columns=[
    "district_id_left", "district_id_right",
    "district_name_left", "district_name_right"
], errors="ignore")



In [242]:
spatis = spatis.drop_duplicates(subset=["id"])

print("Duplicates removed. Records remaining:", len(spatis))


Duplicates removed. Records remaining: 1606


In [243]:

spatis = spatis.replace({None: np.nan})


In [244]:

print(spatis.columns.tolist())
print(spatis.head(3))


['id', 'name', 'brand', 'operator', 'latitude', 'longitude', 'neighborhood_id', 'neighborhood_name', 'openinghours', 'phone', 'website', 'geometry', 'address', 'district_id', 'district_name']
         id            name brand        operator   latitude  longitude  \
0  26867411  Bavaria petrol   NaN  Bavaria Petrol  52.501974  13.294496   
1  29997723            Aral  Aral     Anne Notzke  52.508370  13.280947   
2  63253672       Späti Joe   NaN             NaN  52.499322  13.296118   

  neighborhood_id neighborhood_name                       openinghours phone  \
0            0407          Halensee  Mo-Fr 07:00-22:00; Sa 08:00-22:00   NaN   
1            0405           Westend                               24/7   NaN   
2            0407          Halensee                               24/7   NaN   

                                             website  \
0                                                NaN   
1  https://tankstelle.aral.de/tankstelle/berlin/m...   
2                 

In [245]:
selected_columns = [
    "id",
    "name",
    "brand",
    "operator",
    "latitude",
    "longitude",
    "district_id",
    "district_name",
    "neighborhood_id",
    "neighborhood_name",
    "openinghours",  
    "phone",
    "website",
    "geometry",
    "address"
]



In [246]:
final_cols = [
    "id", "name", "brand", "operator",
    "latitude", "longitude",
    "district_id", "district_name",
    "neighborhood_id", "neighborhood_name",
    "openinghours",   # <-- correct name after renaming
    "phone", "website",
    "geometry",
    "address"
]





In [251]:
# Define the columns you want in the final output
final_cols = [
    "id", "name", "brand", "operator",
    "latitude", "longitude",
    "district_id", "district_name",
    "neighborhood_id", "neighborhood_name",
    "openinghours", "phone", "website",
    "geometry", "address"
]

# Keep only columns that exist in the joined spatis
final_cols = [c for c in final_cols if c in spatis.columns]

# Remove duplicate IDs
spatis = spatis.drop_duplicates(subset=["id"])

# Select the final columns
spatis = spatis[final_cols]

# Check result
spatis.head(5)








,id,name,brand,operator,latitude,longitude,district_id,district_name,neighborhood_id,neighborhood_name,openinghours,phone,website,geometry,address
0,26867411,Bavaria petrol,NaN,Bavaria Petrol,52.501974,13.294496,004,Charlottenburg-Wilmersdorf,0407,Halensee,Mo-Fr 07:00-22:00; Sa 08:00-22:00,NaN,NaN,POINT (13.2945 52.50197),
1,29997723,Aral,Aral,Anne Notzke,52.508370,13.280947,004,Charlottenburg-Wilmersdorf,0405,Westend,24/7,NaN,https://tankstelle.aral.de/tankstelle/berlin/m...,POINT (13.28095 52.50837),
2,63253672,Späti Joe,NaN,NaN,52.499322,13.296118,004,Charlottenburg-Wilmersdorf,0407,Halensee,24/7,NaN,NaN,POINT (13.29612 52.49932),
3,253616592,Mein Markt Pham,NaN,NaN,52.511047,13.462698,002,Friedrichshain-Kreuzberg,0201,Friedrichshain,Mo-Fr 08:00-20:00; Sa 08:00-19:00,NaN,NaN,POINT (13.4627 52.51105),
4,266629404,...nah und gut,EDEKA,Heinz Vollack,52.509209,13.587500,010,Marzahn-Hellersdorf,1003,Kaulsdorf,Mo-Fr 07:00-19:00; Sa 07:00-13:00; PH off,+49 30 5677706,NaN,POINT (13.5875 52.50921),


In [252]:
# ===============================
# Export final SPATIS dataset
# ===============================

# Export GeoJSON with geometry
spatis.to_file("spatis_with_admins.geojson", driver="GeoJSON")

# Export CSV without geometry
spatis.drop(columns="geometry").to_csv("spatis_with_admins.csv", index=False)

print("SPATIS export complete. Records:", len(spatis))


SPATIS export complete. Records: 1606
